# Competencia 1

## Generacion de la clase_ternaria

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/datasets/competencia_01_crudo.csv" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]



In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

ESTABILIZACIÓN POR IPC

In [ ]:
# =================== ESTABILIZACIÓN POR IPC (base 202104 = 100) ===================

library(data.table)

# 1) Tabla IPC ( % ya convertidos a índice con base abril=100)
ipc <- data.table(
  foto_mes = c(202101L, 202102L, 202103L, 202104L, 202105L, 202106L),
  indice   = c(90.7,     92.1,     95.4,     100.0,    103.3,    106.6)
)

# 2) Merge por foto_mes
stopifnot("foto_mes" %in% names(dataset))
dataset <- merge(dataset, ipc, by = "foto_mes", all.x = TRUE, sort = FALSE)

# 3) Deflactor relativo a 202104
dataset[, defl_factor := indice / 100]


# 4) Detectar columnas de montos a deflactar
#    - Montos "generales": empiezan con 'm'
#    - Montos de tarjetas: nombres como 'Master_m...' o 'Visa_m...'
montos_m_prefix <- grep("^m", names(dataset), value = TRUE)
montos_card     <- grep("^(Master|Visa)_m", names(dataset), value = TRUE)

montos_all <- unique(c(montos_m_prefix, montos_card))

# por si alguna coincide pero no es numérica
montos_all <- montos_all[vapply(dataset[, ..montos_all], is.numeric, logical(1))]

# 5) Deflactar IN-PLACE: x := x / defl_factor  (expresado en pesos constantes 202104)
for (cn in montos_all) {
  dataset[, (cn) := get(cn) / pmax(defl_factor, 1e-9)]
}

# 6) (Opcional) dejar rastro
#dataset[, ipc_base := 202104L]


cat("IPC aplicado (base 202104). Columnas deflactadas:", length(montos_all), "\n")
# =================== /ESTABILIZACIÓN POR IPC ===================



In [ ]:
# eliminar columnas de IPC para no laguearlas ni usarlas como features
drop_ipc <- intersect(c("indice","defl_factor","ipc_base"), names(dataset))
if (length(drop_ipc)) dataset[, (drop_ipc) := NULL]

In [ ]:
ncol(dataset)
colnames(dataset)

SAC

In [ ]:
# ====== AJUSTE DE AGUINALDO  ======

evento_aguinaldo <- 202106L
tau <- 0.45

# señales de haberes
dataset[, payroll_monto := fcoalesce(mpayroll, 0) + fcoalesce(mpayroll2, 0)]
dataset[, payroll_freq  := fcoalesce(cpayroll_trx, 0L) + fcoalesce(cpayroll2_trx, 0L)]
dataset[, has_payroll   := as.integer(payroll_monto > 0 | payroll_freq > 0)]

# lags para sueldos
dataset[, `:=`(
  mpayroll_lag1       = shift(mpayroll, 1L),
  mpayroll2_lag1      = shift(mpayroll2, 1L),
  pay_monto_lag1      = shift(payroll_monto, 1L),
  cpayroll_trx_lag1   = shift(cpayroll_trx, 1L),
  cpayroll2_trx_lag1  = shift(cpayroll2_trx, 1L)
), by = numero_de_cliente]

# rolling max 6m PREVIO (excluye el mes actual)
dataset[, pay_max6_prev := frollapply(shift(payroll_monto, 1L),
                                      6L, max, align = "right", na.rm = TRUE),
        by = numero_de_cliente]

# SAC teórico en base a histórico previo
dataset[, sac_teorico := 0.5 * pay_max6_prev]

# salto vs mayo
dataset[, inc_vs_may := payroll_monto - pay_monto_lag1]

# detectar SAC solo si hay haberes y el salto es “≈ media remuneración previa”
dataset[, sac_flag := (foto_mes == evento_aguinaldo) &
                      ((fcoalesce(mpayroll, 0) + fcoalesce(mpayroll2, 0)) > 0) &
                      is.finite(inc_vs_may) & is.finite(pay_max6_prev) &
                      (inc_vs_may >= tau * pay_max6_prev)]

# 1) Ajuste mpayroll / mpayroll2 proporcional sin crear columnas extra
dataset[sac_flag == TRUE, `:=`(
  mpayroll  = {
    total <- pmax(mpayroll + mpayroll2, 1e-9)
    desc  <- pmin(sac_teorico, payroll_monto)
    pmax(0, mpayroll  - desc * (mpayroll  / total))
  },
  mpayroll2 = {
    total <- pmax(mpayroll + mpayroll2, 1e-9)
    desc  <- pmin(sac_teorico, payroll_monto)
    pmax(0, mpayroll2 - desc * (mpayroll2 / total))
  }
)]

# 2) Ajuste de saldo (usa lag/roll existentes, pero no crea columnas nuevas)
# cuantiles del TRAIN para cap
idx_train <- dataset$foto_mes %in% c(202101L, 202102L, 202103L, 202104L)
q1_s  <- as.numeric(quantile(dataset[idx_train, mcuentas_saldo], 0.01, na.rm = TRUE, type = 7))
q99_s <- as.numeric(quantile(dataset[idx_train, mcuentas_saldo], 0.99, na.rm = TRUE, type = 7))


if (!"mcuentas_saldo_lag1" %in% names(dataset))
  dataset[, mcuentas_saldo_lag1 := shift(mcuentas_saldo, 1L), by = numero_de_cliente]
if (!"mcuentas_saldo_roll_mean3" %in% names(dataset))
  dataset[, mcuentas_saldo_roll_mean3 := frollmean(mcuentas_saldo, 3L, align = "right"), by = numero_de_cliente]

dataset[sac_flag == TRUE & foto_mes == evento_aguinaldo,
        mcuentas_saldo := {
          base <- fifelse(!is.na(mcuentas_saldo_lag1), mcuentas_saldo_lag1, mcuentas_saldo_roll_mean3)
          inc  <- pmax(0, mcuentas_saldo - base)
          desc <- pmin(sac_teorico, payroll_monto)
          nuevo <- mcuentas_saldo - pmin(inc, desc)
          pmin(q99_s, pmax(q1_s, nuevo))
        }]


# Limpieza mínima de auxiliares de este bloque
dataset[, `:=`(inc_vs_may = NULL, sac_teorico = NULL)]
# ====== /SAC ======

  # --- limpiar auxiliares del bloque SAC ---
cols_temp <- c(
  "payroll_monto","payroll_freq","has_payroll","pay_monto_lag1",
  "pay_max6","sac_teorico","inc_vs_may","sac_flag","mcuentas_saldo_roll_mean3","pay_max6_prev"
)
cols_temp <- intersect(cols_temp, names(dataset))
if (length(cols_temp)) dataset[, (cols_temp) := NULL]

aux <- intersect(c("mpayroll_lag1","mpayroll2_lag1","pay_monto_lag1",
                   "cpayroll_trx_lag1","cpayroll2_trx_lag1",
                   "mcuentas_saldo_lag1","mcuentas_saldo_roll_mean3",
                   "pay_max6_prev","inc_vs_may","sac_teorico","sac_flag"),
                 names(dataset))
if (length(aux)) dataset[, (aux) := NULL]

In [ ]:
ncol(dataset)
colnames(dataset)

LAGS

In [ ]:
library(data.table)
setorder(dataset, numero_de_cliente, foto_mes)
setDTthreads(0L)




# Feature Engineering Historico
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente","foto_mes","clase_ternaria","clase01",
                  "training","azar","cliente_antiguedad","cliente_edad",
                  "Master_status","Visa_status")
) )

#Prealocar (ahora que ya sabemos cuántas lagueables hay)
alloc.col(dataset, ncol(dataset) + 4L * length(cols_lagueables) + 50L)



dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags de orden 1
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}





# Limpiar auxiliares (una sola vez)
dataset[, c("mes_id","mes_id_lag1","mes_id_lag2","cons1","cons2") := NULL]
gc()

# sanity check
faltan_lag2 <- setdiff(paste0(cols_lagueables, "_lag2"), names(dataset))
if (length(faltan_lag2)) stop("No se crearon estas _lag2: ", paste(faltan_lag2, collapse=", "))

cat(" LAGS/DELTAS OK — lag1:", length(cols_lagueables),
    " lag2:", length(cols_lagueables),
    " delta1:", length(cols_lagueables),
    " delta2:", length(cols_lagueables), "\n")

In [ ]:
ncol(dataset)
colnames(dataset)

In [ ]:
fwrite( dataset,
    file =  "/content/datasets/competencia_01.csv.gz",
    sep = ","
)

### Optimizacion Hiperparámetros

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

### Carga de Librerias

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

### Definicion de Parametros

In [ ]:
PARAM <- list()
PARAM$experimento <- "TP1_11_4"
PARAM$semilla_primigenia <- 102191


In [ ]:
# training y future
# BO: 202103 completo + 202101, 202102 solo BAJAS
PARAM$train_full    <- c(202103L)
PARAM$train_posonly <- c(202101L, 202102L)

PARAM$train <- c(202101, 202102,202103)
PARAM$valid <- c(202104)
PARAM$train_final <- c(202101, 202102, 202103, 202104)
PARAM$future <- c(202106)
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- seq(6000, 19000, by= 250)

In [ ]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 1.0

In [ ]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 1200,
  learning_rate= 0.02,
  feature_fraction= 0.5,
  num_leaves= 750,
  min_data_in_leaf= 5000
)


In [ ]:
# Aqui se cargan los bordes de los hiperparametros de la BO
PARAM$hyperparametertuning$hs <- makeParamSet(
  makeIntegerParam("num_iterations", lower= 8L, upper= 2048L),
  makeNumericParam("learning_rate", lower= 0.01, upper= 0.3),
  makeNumericParam("feature_fraction", lower= 0.1, upper= 1.0),
  makeIntegerParam("num_leaves", lower= 8L, upper= 2048L),
  makeIntegerParam("min_data_in_leaf", lower= 1L, upper= 8000L),
  #makeNumericParam("min_gain_to_split", lower=0.0, upper=5.0),
  makeNumericParam("lambda_l1", lower=0.0, upper=50.0),
  makeNumericParam("lambda_l2", lower=0.0, upper=50.0)

)

In [ ]:
PARAM$hyperparametertuning$iteraciones <- 50 # iteraciones bayesianas

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [ ]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [ ]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 780000, -20000))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 780000, -20000))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 780000, -20000))]

  prealidad[, predicted:=NULL]
  return( res )
}

### Preprocesamiento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz", stringsAsFactors= TRUE)

In [ ]:
idx_train <- (dataset$foto_mes %in% c(PARAM$train_full, PARAM$train_posonly))
dataset_train <- dataset[idx_train]

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]


dataset_train[
  (foto_mes %in% PARAM$train_full) |
  (foto_mes %in% PARAM$train_posonly &  clase01==1L),
  training := 1L
]

In [ ]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("numero_de_cliente", "clase_ternaria", "clase01", "azar", "training")
)


In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

Configuracion Bayesian Optimization

In [ ]:
# En el argumento x llegan los parmaetros de la bayesiana
#  devuelve la AUC en cross validation del modelo entrenado

EstimarGanancia_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
  nrounds_es <- if (!is.null(param_completo$num_iterations)) as.integer(param_completo$num_iterations) else 1200L
  es_rounds  <- max(50L, round(nrounds_es * 0.15))
  param_completo$data_random_seed <- NULL

  # entreno LightGBM
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,
    stratified= TRUE,
    param= param_completo,
    nrounds = nrounds_es,
    early_stopping_rounds = es_rounds,
    verbose = -1
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y"), " AUC ", AUC)

  return(AUC)
}


In [ ]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- EstimarGanancia_AUC_lightgbm # la funcion que voy a maximizar

configureMlr(show.learner.output= FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo

obj.fun <- makeSingleObjectiveFunction(
  fn= funcion_optimizar, # la funcion que voy a maximizar
  minimize= FALSE, # estoy Maximizando la ganancia
  noisy= TRUE,
  par.set= PARAM$hyperparametertuning$hs, # definido al comienzo del programa
  has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
  save.on.disk.at.time= 600, # se graba cada 600 segundos
  save.file.path= kbayesiana
) # se graba cada 600 segundos

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
  ctrl,
  iters= PARAM$hyperparametertuning$iteraciones
) # cantidad de iteraciones

# defino el método estandar para la creacion de los puntos iniciales,
# los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

# establezco la funcion que busca el maximo
surr.km <- makeLearner(
  "regr.km",
  predict.type= "se",
  covtype= "matern3_2",
  control= list(trace= TRUE)
)


Corrida Bayesian Optimization

In [ ]:
# inicio la optimizacion bayesiana, retomando si ya existe
# es la celda mas lenta de todo el notebook

if (!file.exists(kbayesiana)) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue(kbayesiana) # retomo en caso que ya exista
}

In [ ]:

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)
colnames( tb_bayesiana)

In [ ]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

tb_bayesiana[, iter := .I]

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file= "BO_log.txt",
  sep= "\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  setdiff(colnames(tb_bayesiana),
    c("y","dob","eol","error.message","exec.time","ei","error.model",
      "train.time","prop.type","propose.time","se","mean","iter")),
  with= FALSE
]


PARAM$out$lgbm$y <- tb_bayesiana[1, y]


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
print(PARAM$out$lgbm$mejores_hiperparametros)
print(PARAM$out$lgbm$y)

## VALID

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train <- dataset[foto_mes %in% PARAM$train]
dataset_train[,.N,clase_ternaria]

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain_valid <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)



#### Hyperparameters

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [ ]:
# Entrenar en TRAIN y predecir en VALID

modelo_valid <- lgb.train(
  data= dtrain_valid,
  param= param_normalizado
)


In [ ]:
# Conjunto de validación
valid_set <- dataset[ foto_mes %in% PARAM$valid ]

pred_valid <- predict(
  modelo_valid,
  data.matrix(valid_set[, campos_buenos, with = FALSE])
)

In [ ]:
# 4) Tabla valid con probas y ganancia por fila
tb_valid <- valid_set[, .(numero_de_cliente, clase_ternaria)]
tb_valid[, prob := pred_valid]
setorder(tb_valid, -prob)

In [ ]:
# Ganancia por “invitar” fila a fila (ordenada por prob desc)
tb_valid[, gain_row := fifelse(clase_ternaria=="BAJA+2", 780000, -20000)]
tb_valid[, gain_cum := cumsum(gain_row)]

In [ ]:
# 5) Imprimir TODOS los cortes K y elegir el mejor
K_grid <- PARAM$cortes                     # ej. seq(6000, 19000, by=250)
k_eff  <- pmin(K_grid, nrow(tb_valid))
res_k  <- data.table(K = K_grid, Ganancia = tb_valid$gain_cum[k_eff])

print(res_k)                               # <<--- todos los K con su ganancia

In [ ]:
best_idx  <- which.max(res_k$Ganancia)
K_offline <- res_k$K[best_idx]
G_offline <- res_k$Ganancia[best_idx]
cat(">> K_offline (valid) =", K_offline, "| ganancia =", G_offline, "\n")
# (opcional) guardar tabla K vs ganancia
fwrite(res_k, file = "ganancia_por_K_valid.csv")



In [ ]:
# 6) Importancia y guardado del modelo de valid (por prolijidad)
tb_importancia <- as.data.table(lgb.importance(modelo_valid))
fwrite(tb_importancia, file = "impo_valid.txt", sep = "\t")
lgb.save(modelo_valid, "modelo_valid.txt")

In [ ]:
write_yaml( PARAM, file="PARAM_valid.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

# FINAL

Final Training Dataset


In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz", stringsAsFactors= TRUE)

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset[,clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L) ]
#dataset_train[,clase01 := ifelse(clase_ternaria %in% c("BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train_final <- dataset[foto_mes %in% PARAM$train_final]
dataset_train_final[,.N,clase_ternaria]

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train_final[, campos_buenos, with= FALSE]),
  label= dataset_train_final[, clase01]
)

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos, PARAM$out$lgbm$mejores_hiperparametros)

param_final

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)
param_normalizado$bagging_fraction <- 0.9
param_normalizado$bagging_freq     <- 1L


In [ ]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes %in% PARAM$future]

In [ ]:
# --- ensemble multi-seed ---
SEEDS <- c(102191, 230101, 999199, 878787, 761177, 161803, 550051, 420420, 314159, 271828)

preds_list <- vector("list", length(SEEDS))
impo_list  <- vector("list", length(SEEDS))

for (i in seq_along(SEEDS)) {
  si <- SEEDS[i]
  p  <- modifyList(param_normalizado, list(seed = si, bagging_seed = si, feature_fraction_seed = si))
  modelo_i <- lgb.train(data = dtrain_final, param = p)

  # guardo importancia por seed (luego promedio)
  impo_list[[i]] <- as.data.table(lgb.importance(modelo_i))[ , seed := si ]

  # guardo el modelo de cada seed
  lgb.save(modelo_i, sprintf("modelo_seed_%d.txt", si))

  # predicción de esta seed
  preds_list[[i]] <- predict(modelo_i, data.matrix(dfuture[, campos_buenos, with = FALSE]))
  rm(modelo_i); gc()
}

In [ ]:
# promedio
pred_mat <- do.call(cbind, preds_list)

# tabla de predicción final (promedio de semillas)
tb_prediccion <- dfuture[, .(numero_de_cliente, foto_mes)]
tb_prediccion[, prob := rowMeans(pred_mat)]   # o: apply(pred_mat, 1, median)

In [ ]:
# Importancia PROMEDIO del ensemble
tb_importancia <- rbindlist(impo_list, fill = TRUE)[
  , .(
      gain_mean   = mean(Gain, na.rm = TRUE),
      gain_median = median(Gain, na.rm = TRUE),
      freq_mean   = mean(Frequency, na.rm = TRUE)
    ),
  by = Feature
][order(-gain_mean)]

fwrite(tb_importancia, file = "impo_ensemble.txt", sep = "\t")
fwrite(tb_prediccion,  file = "prediccion.txt",     sep = "\t")

Aplico el modelo final a los datos del futuro

In [ ]:
# inicilizo el dataset  drealidad
drealidad <- realidad_inicializar( dfuture, PARAM)

Kaggle Competition Submit

In [ ]:
PARAM$cortes

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  res <- realidad_evaluar( drealidad, tb_prediccion)

  options(scipen = 999)
  cat( "Envios=", envios, "\t",
    " TOTAL=", res$total,
    "  Public=", res$public,
    " Private=", res$private,
    "\n",
    sep= ""
  )

}

In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")